# 🏆 CyberShelf — XGBoost | 2× T4 GPU

**Только XGBoost** — быстро и качественно (~2 часа на 2× T4)
- 3 фолда
- learning_rate = 0.1
- num_boost_round = 500
- Оба GPU используются: фолды распределяются между GPU 0 и GPU 1

In [2]:
!pip install xgboost --quiet

In [3]:
import numpy as np
import pandas as pd
import gc, os, time, subprocess, warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score


def show_ram():
    import psutil
    ram = psutil.virtual_memory()
    print(f'RAM: {ram.used/1024**3:.1f}GB / {ram.total/1024**3:.1f}GB  ({ram.percent}%)')

def show_gpu():
    try:
        r = subprocess.run(
            ['nvidia-smi',
             '--query-gpu=index,name,memory.used,memory.total,utilization.gpu',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True)
        for line in r.stdout.strip().split('\n'):
            idx, name, mu, mt, util = [x.strip() for x in line.split(',')]
            print(f'  GPU {idx} [{name}]  VRAM: {mu}/{mt} MiB  Util: {util}%')
    except Exception as e:
        print('nvidia-smi недоступен:', e)

print('✅ Импорты загружены')
show_gpu()
show_ram()

✅ Импорты загружены
  GPU 0 [Tesla T4]  VRAM: 0/15360 MiB  Util: 0%
  GPU 1 [Tesla T4]  VRAM: 0/15360 MiB  Util: 0%
RAM: 0.7GB / 31.4GB  (3.8%)


In [4]:
BASE_URL = 'https://storage.yandexcloud.net/data-fusion-2026/2%20CyberShelf'
!wget -q "{BASE_URL}/train_main_features.parquet"
!wget -q "{BASE_URL}/test_main_features.parquet"
!wget -q "{BASE_URL}/train_target.parquet"
!wget -q "{BASE_URL}/sample_submit.parquet"
print('✅ Данные загружены!')

^C
✅ Данные загружены!


In [6]:
train         = pd.read_parquet('train_main_features.parquet')
test          = pd.read_parquet('test_main_features.parquet')
target        = pd.read_parquet('train_target.parquet')
sample_submit = pd.read_parquet('sample_submit.parquet')

print('Train shape:', train.shape)
print('Test  shape:', test.shape)
print('Target cols:', target.shape[1] - 1)
show_ram()

ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

In [7]:
cat_feature_names = [col for col in train.columns if col.startswith('cat_feature')]
target_columns    = [col for col in target.columns if col.startswith('target')]
feature_cols      = [col for col in train.columns if col != 'customer_id']

print(f'Признаков:      {len(feature_cols)}')
print(f'Кат. признаков: {len(cat_feature_names)}')
print(f'Таргетов:       {len(target_columns)}')

def reduce_memory(df):
    for col in df.select_dtypes('float64').columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes('int64').columns:
        df[col] = df[col].astype('int32')
    return df

train  = reduce_memory(train)
test   = reduce_memory(test)
target = reduce_memory(target)

# XGBoost не принимает category — сразу кодируем в int
for col in cat_feature_names:
    train[col] = train[col].astype('category').cat.codes
    test[col]  = test[col].astype('category').cat.codes

X      = train[feature_cols]
y      = target[target_columns]
X_test = test[feature_cols]

gc.collect()
print('\n✅ Данные готовы')
show_ram()

NameError: name 'train' is not defined

In [ ]:
N_FOLDS      = 3
RANDOM_STATE = 42

try:
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=index,name', '--format=csv,noheader'],
        capture_output=True, text=True)
    gpu_list = [l.strip() for l in r.stdout.strip().split('\n') if l.strip()]
    N_GPUS = len(gpu_list)
    print(f'Найдено GPU: {N_GPUS}')
    for g in gpu_list: print(' ', g)
except Exception:
    N_GPUS = 0
    print('GPU не найдены → CPU')

xgb_params = {
    'objective':        'binary:logistic',
    'eval_metric':      'auc',
    'learning_rate':    0.1,
    'max_depth':        5,
    'subsample':        0.6,
    'colsample_bytree': 0.6,
    'min_child_weight': 50,
    'reg_alpha':        0.1,
    'reg_lambda':       0.1,
    'device':           'cuda:0',  # CUDA_VISIBLE_DEVICES выберет нужный GPU
    'random_state':     RANDOM_STATE,
    'verbosity':        0,
    'n_jobs':           4,
}

print(f'\nXGBoost device: {xgb_params["device"]}')
print('✅ Конфигурация готова')

In [ ]:
%%writefile workers.py
import os
import gc
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score


def train_xgb_gpu0(args):
    # Обучает нечётные таргеты на GPU 0
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    import xgboost as xgb
    return _train_xgb(args)


def train_xgb_gpu1(args):
    # Обучает чётные таргеты на GPU 1
    os.environ['CUDA_VISIBLE_DEVICES'] = '1'
    import xgboost as xgb
    return _train_xgb(args)


def _train_xgb(args):
    import xgboost as xgb

    X, y_df, X_test, target_columns, params, n_folds, seed = args

    n_tr, n_te, n_t = len(X), len(X_test), len(target_columns)
    oof   = np.zeros((n_tr, n_t), dtype='float32')
    test  = np.zeros((n_te, n_t), dtype='float32')
    kf    = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    dtest = xgb.DMatrix(X_test)

    for i, tc in enumerate(target_columns):
        y_col = y_df[tc].values
        tf    = np.zeros(n_te, dtype='float32')
        for tr_idx, val_idx in kf.split(X):
            d_tr  = xgb.DMatrix(X.iloc[tr_idx],  label=y_col[tr_idx])
            d_val = xgb.DMatrix(X.iloc[val_idx], label=y_col[val_idx])
            m = xgb.train(
                params, d_tr,
                num_boost_round=500,
                evals=[(d_val, 'val')],
                early_stopping_rounds=30,
                verbose_eval=False,
            )
            oof[val_idx, i] += m.predict(d_val).astype('float32')
            tf              += m.predict(dtest).astype('float32') / n_folds
            del m, d_tr, d_val
            gc.collect()
        test[:, i] = tf

    auc = float(np.mean([
        roc_auc_score(y_df[tc], oof[:, i])
        for i, tc in enumerate(target_columns)
    ]))
    return oof, test, auc

In [ ]:
import importlib, sys
if 'workers' in sys.modules:
    importlib.reload(sys.modules['workers'])
from workers import train_xgb_gpu0, train_xgb_gpu1
print('✅ workers.py импортирован')

In [ ]:
import multiprocessing as mp

# Делим 41 таргет пополам между двумя GPU
half      = len(target_columns) // 2
targets_0 = target_columns[:half]       # первые 20 таргетов → GPU 0
targets_1 = target_columns[half:]       # остальные 21 таргет → GPU 1

args_gpu0 = (X, y[targets_0], X_test, targets_0, xgb_params, N_FOLDS, RANDOM_STATE)
args_gpu1 = (X, y[targets_1], X_test, targets_1, xgb_params, N_FOLDS, RANDOM_STATE)

print('=' * 60)
if N_GPUS >= 2:
    print(f'  GPU 0 → {len(targets_0)} таргетов')
    print(f'  GPU 1 → {len(targets_1)} таргетов')
    print('  Параллельно!')
else:
    print('  1 GPU → последовательное обучение')
print('=' * 60)

t0 = time.time()

if N_GPUS >= 2:
    ctx = mp.get_context('spawn')
    with ctx.Pool(processes=2) as pool:
        res0 = pool.apply_async(train_xgb_gpu0, (args_gpu0,))
        res1 = pool.apply_async(train_xgb_gpu1, (args_gpu1,))
        oof0, test0, auc0 = res0.get()
        oof1, test1, auc1 = res1.get()
else:
    print('[1/2] Таргеты 0...')
    oof0, test0, auc0 = train_xgb_gpu0(args_gpu0)
    print('[2/2] Таргеты 1...')
    oof1, test1, auc1 = train_xgb_gpu1(args_gpu1)

# Склеиваем результаты обоих GPU обратно
oof_xgb  = np.concatenate([oof0,  oof1],  axis=1)
test_xgb = np.concatenate([test0, test1], axis=1)
xgb_auc  = np.mean([auc0, auc1])

print(f'\nОбщее время: {time.time()-t0:.0f}s')
print(f'📊 OOF AUC — GPU 0 ({len(targets_0)} таргетов): {auc0:.4f}')
print(f'📊 OOF AUC — GPU 1 ({len(targets_1)} таргетов): {auc1:.4f}')
print(f'📊 OOF AUC — Итого: {xgb_auc:.4f}')
show_ram()
show_gpu()

In [ ]:
submit.columns = ['customer_id'] + [col.replace('target_', 'predict_') for col in submit.columns[1:]]

submit.to_parquet('/kaggle/working/submission.parquet', index=False)
print('✅ Готово!')
print(submit.columns.tolist())  # проверка

all_targets = targets_0 + targets_1

submit = pd.DataFrame(test_xgb, columns=all_targets)
submit = submit[target_columns]  # восстанавливаем оригинальный порядок
submit.insert(0, 'customer_id', test['customer_id'].values)

print('Форма сабмита:            ', submit.shape)
print('Колонки == sample_submit: ', list(submit.columns) == list(sample_submit.columns))
print('Пропуски:                 ', submit[target_columns].isna().sum().sum())
print(submit.head(3))

submit.to_parquet('submission.parquet', index=False)
print('\n✅ submission.parquet сохранён!')

In [ ]:
import pandas as pd

sub = pd.read_parquet('/kaggle/working/submission.parquet')

# Переименование target_* → predict_*
sub = sub.rename(columns=lambda col: col.replace('target_', 'predict_') if col != 'customer_id' else col)

# Приводим все колонки кроме customer_id к float64
predict_cols = [col for col in sub.columns if col != 'customer_id']
sub[predict_cols] = sub[predict_cols].astype('float64')

# Сохраняем
sub.to_parquet('/kaggle/working/submission.parquet', index=False)
print("✅ Готово!")
print(sub.dtypes)

In [ ]:
predict_cols = [col for col in sub.columns if col != 'customer_id']
sub[predict_cols] = sub[predict_cols].astype('float64')

In [ ]:
pd.read_parquet('/kaggle/working/submission.parquet').columns.tolist()

In [ ]:
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import HfApi
#
# hf_token = UserSecretsClient().get_secret('HF_TOKEN')
# api      = HfApi()
# repo_id  = 'ВАШ_НИКНЕЙМ/cybershelf-submission'
# api.create_repo(repo_id, repo_type='dataset', private=True, token=hf_token, exist_ok=True)
# api.upload_file(
#     path_or_fileobj='submission.parquet',
#     path_in_repo='submission.parquet',
#     repo_id=repo_id, repo_type='dataset', token=hf_token,
# )
# print(f'✅ Загружено: https://huggingface.co/datasets/{repo_id}')

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("Dinaro")

In [ ]:
repo_id = "Dinar1809/vtb"

model.module.push_to_hub(repo_id, private=True, token = api_key)
processor.push_to_hub(repo_id,     private=True, token = api_key)
new_tokenizer.push_to_hub(repo_id,     private=True, token = api_key)

print(f"Готово! https://huggingface.co/PavelDuriy/trocr-handwriting-ru-v2")

In [ ]:
import os
path = '/kaggle/working/submission.parquet'
if os.path.exists(path):
    size = os.path.getsize(path) / 1024**2
    print(f'✅ Файл есть, размер: {size:.1f} MB')
else:
    print('❌ Файла нет — запусти ячейку сохранения')

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("Dinaro")

In [ ]:
repo_id = "Dinar1809/vtb"

model.module.push_to_hub(repo_id, private=True, token = api_key)
processor.push_to_hub(repo_id,     private=True, token = api_key)
new_tokenizer.push_to_hub(repo_id,     private=True, token = api_key)

print(f"Готово! https://huggingface.co/PavelDuriy/trocr-handwriting-ru-v2")